In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import mlflow
import mlflow.pyfunc
import re
from bs4 import BeautifulSoup

# --------------------------------------------------
# ✅ INIT API
# --------------------------------------------------

app = FastAPI(title="Rakuten Product Classification API")

# --------------------------------------------------
# ✅ MLflow CONFIG
# --------------------------------------------------

mlflow.set_tracking_uri("file:///C:/Users/user/Rakuten-Challenge/mlruns")

# ✅ modèle en production (pipeline complet recommandé)
model = mlflow.pyfunc.load_model("models:/rakuten_model@prod")

# --------------------------------------------------
# ✅ DATA MODEL
# --------------------------------------------------

class Product(BaseModel):
    designation: str
    description: str | None = ""

# --------------------------------------------------
# ✅ CLEANING FUNCTION
# --------------------------------------------------

def clean_text(text: str) -> str:
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --------------------------------------------------
# ✅ HEALTH CHECK
# --------------------------------------------------

@app.get("/health")
def health():
    return {"status": "ok"}

# --------------------------------------------------
# ✅ SINGLE PREDICTION
# --------------------------------------------------

@app.post("/predict")
def predict(product: Product):
    if not product.designation:
        raise HTTPException(status_code=400, detail="Designation is required")

    text = clean_text(product.designation + " " + (product.description or ""))

    try:
        prediction = model.predict([text])[0]
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    return {"prediction": int(prediction)}

# --------------------------------------------------
# ✅ BATCH PREDICTION
# --------------------------------------------------

@app.post("/predict_batch")
def predict_batch(products: List[Product]):
    if len(products) == 0:
        raise HTTPException(status_code=400, detail="Empty input list")

    texts = [
        clean_text(p.designation + " " + (p.description or ""))
        for p in products
    ]

    try:
        predictions = model.predict(texts)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    return {"predictions": predictions.tolist()}
